# Agente 1 — Pipeline base: NL → `FinanceQuery` → parámetros de `yfinance`

Este notebook documenta y construye la **primera parte** del sistema del TFM: el agente que recibe una petición en lenguaje natural y la transforma en una **consulta financiera normalizada, validada y lista para descargarse** con `yfinance`.

## Objetivo del notebook

En esta fase no se busca todavía responder al usuario con una explicación o análisis financiero final. El objetivo es construir un **objeto intermedio estable** que actúe como contrato entre:

- la **frase humana** que escribe el usuario;
- y la **petición técnica** que necesita `yf.download(...)`.

La idea central es que el primer agente **no debe producir directamente una respuesta final**, sino generar primero una estructura intermedia del tipo:

~~~python
FinanceQuery(...)
~~~

Ese objeto permite desacoplar el lenguaje natural de la descarga real de datos, facilitando la trazabilidad, la validación y la depuración del sistema.

## Flujo conceptual implementado

1. **Interpretación**  
   Convierte lenguaje natural en una consulta financiera estructurada, detectando intención, activos, fechas y granularidad temporal.

2. **Resolución**  
   Transforma nombres y alias como `Nvidia`, `S&P 500`, `oro` o `Bitcoin` en símbolos compatibles con Yahoo Finance.

3. **Normalización temporal**  
   Convierte expresiones humanas de tiempo en `period` o en `start/end`.

4. **Validación**  
   Comprueba si la petición tiene sentido y si respeta restricciones de `yfinance`, por ejemplo la compatibilidad entre ticker, rango temporal e intervalo.

5. **Construcción de la petición**  
   Genera los argumentos finales que se pasarían a `yf.download(...)`.

6. **Descarga**  
   Obtiene el dataframe bruto con los datos históricos solicitados.

7. **Postproceso**  
   Deja preparado el guardado a CSV, la metadata asociada a la consulta y las posibles advertencias detectadas durante el proceso.

## Resultado esperado

A partir de una consulta como:

> `Cuánto ha crecido Nvidia en 5 años`

el pipeline debería construir una representación intermedia equivalente a:

~~~json
{
  "intent": "price_growth",
  "resolved_tickers": ["NVDA"],
  "period": "5y",
  "interval": "1d"
}
~~~

y, a partir de ella, generar unos parámetros de descarga válidos para `yfinance`.

## En qué se apoya esta versión

Esta primera versión del agente se apoya en:

- un **universo curado de tickers** cargado desde `important_yfinance_tickers.txt`;
- alias manuales frecuentes como `oro`, `bitcoin`, `S&P 500` o `nvidia`;
- reglas transparentes, controlables y fáciles de depurar;
- ejemplos de prueba para validar el pipeline antes de activar la descarga real.

## Qué se deja para una siguiente iteración

Como evolución futura del sistema, se deja abierta la incorporación de:

- resolución dinámica online mediante `Search` y `Lookup` para activos fuera del catálogo;
- desambiguación más fina en activos ambiguos;
- una capa LLM opcional para consultas más abiertas o complejas.

In [1]:
# Si te falta alguna librería en Colab, descomenta esta celda:
# !pip install -q yfinance pandas

import re
import json
import unicodedata
from dataclasses import dataclass, asdict, field
from datetime import date, datetime
from pathlib import Path
from typing import List, Optional, Dict, Tuple

import pandas as pd

TODAY = date.today()
BASE_DIR = Path(".")
TICKER_CATALOG_PATH = BASE_DIR / "important_yfinance_tickers.txt"
EXPORT_DIR = BASE_DIR / "exports"
EXPORT_DIR.mkdir(exist_ok=True, parents=True)

pd.set_option("display.max_colwidth", 120)

## 1. Modelo intermedio: `FinanceQuery`

En vez de conectar directamente el lenguaje natural con `yfinance`, introducimos una **representación intermedia explícita**.

### ¿Por qué es importante?

Porque separa dos niveles del problema:

- **nivel semántico**: entender qué quiere el usuario;
- **nivel técnico**: saber cómo invocar `yf.download(...)`.

### Qué modelamos

Se usan dos dataclasses:

- `AssetResolution`: representa cómo un texto detectado en la frase se ha resuelto a un ticker concreto.
- `FinanceQuery`: representa la consulta completa ya normalizada.

### Ventajas de este diseño

- Hace el sistema más **explicable** en el TFM.
- Permite **validar** antes de descargar.
- Facilita guardar **metadata y advertencias**.
- Aísla errores: si algo falla, sabemos si ha fallado la interpretación o la descarga.

In [2]:
# ============================================================
# Definición del modelo intermedio
# ------------------------------------------------------------
# AssetResolution:
#   Representa cómo una mención textual concreta se resuelve
#   a un ticker de Yahoo Finance.
#
# FinanceQuery:
#   Es el contrato principal del primer agente. Contiene
#   activos, rango temporal, intervalo, configuración de
#   descarga y advertencias.
# ============================================================

@dataclass
class AssetResolution:
    raw_text: str
    matched_text: str
    ticker: str
    asset_type: Optional[str] = None
    description: Optional[str] = None
    source: str = "catalog"
    confidence: float = 1.0

@dataclass
class FinanceQuery:
    original_query: str
    intent: str
    assets_raw: List[str]
    assets_resolved: List[AssetResolution]
    interval: str = "1d"
    start: Optional[str] = None
    end: Optional[str] = None
    period: Optional[str] = None
    group_by: str = "ticker"
    auto_adjust: bool = False
    threads: bool = True
    progress: bool = False
    warnings: List[str] = field(default_factory=list)
    needs_clarification: bool = False

    def resolved_tickers(self) -> List[str]:
        return [a.ticker for a in self.assets_resolved]

    def to_download_params(self) -> Dict:
        params = {
            "tickers": self.resolved_tickers(),
            "interval": self.interval,
            "group_by": self.group_by,
            "auto_adjust": self.auto_adjust,
            "threads": self.threads,
            "progress": self.progress,
        }
        if self.period:
            params["period"] = self.period
        else:
            params["start"] = self.start
            params["end"] = self.end
        return params

    def to_dict(self) -> Dict:
        d = asdict(self)
        d["resolved_tickers"] = self.resolved_tickers()
        return d

## 2. Cargar catálogo curado

Antes de intentar resolver cualquier activo, cargamos una **base local curada** de tickers relevantes.

### Idea de diseño

En esta fase no usamos todavía un resolvedor online abierto.  
Trabajamos con un catálogo controlado para validar bien la arquitectura del agente.

### Qué aporta este catálogo

- Un conjunto conocido de tickers válidos.
- Su tipo de activo (`stock`, `index`, `crypto`, `future`, etc.).
- Una descripción legible que sirve para generar alias.

### Por qué es útil

El catálogo permite combinar dos cosas:

- **precisión**: sabemos qué símbolos existen en nuestro universo de prueba;
- **trazabilidad**: podemos justificar por qué una palabra se resolvió a un ticker.

In [3]:
# ============================================================
# Normalización de texto y carga del catálogo
# ------------------------------------------------------------
# normalize_text():
#   Limpia texto libre para hacerlo comparable:
#   - minúsculas
#   - sin tildes
#   - normalización de símbolos
#
# load_ticker_catalog():
#   Lee el fichero local de tickers y lo convierte en un
#   DataFrame con ticker, tipo de activo y descripción.
# ============================================================

def normalize_text(text: str) -> str:
    text = text.strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace("&", " and ")
    text = text.replace("/", " / ")
    text = re.sub(r"[^a-z0-9^=./+\-\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def load_ticker_catalog(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or " - " not in line:
                continue
            parts = [p.strip() for p in line.split(" - ")]
            rows.append(
                {
                    "ticker": parts[0],
                    "asset_type": parts[1] if len(parts) > 1 else None,
                    "description": " - ".join(parts[2:]) if len(parts) > 2 else None,
                }
            )
    return pd.DataFrame(rows).drop_duplicates(subset=["ticker"]).reset_index(drop=True)

catalog_df = load_ticker_catalog(TICKER_CATALOG_PATH)
print(catalog_df)


    ticker asset_type               description
0    ^GSPC      Index                   S&P 500
1     ^DJI      Index                 Dow Jones
2    ^IXIC      Index          Nasdaq Composite
3     ^RUT      Index              Russell 2000
4     ^VIX      Index     CBOE Volatility Index
..     ...        ...                       ...
183   VRTX      Stock    Vertex Pharmaceuticals
184     VZ      Stock                   Verizon
185    WBA      Stock  Walgreens Boots Alliance
186     WM      Stock          Waste Management
187    ZTS      Stock                    Zoetis

[188 rows x 3 columns]


## 3. Alias manuales y alias derivados del catálogo

Aquí construimos la capa que permite pasar de nombres humanos a símbolos financieros.

### Dos fuentes de alias

1. **Alias manuales**  
   Son equivalencias curadas a mano para casos frecuentes:
   - `nvidia` → `NVDA`
   - `bitcoin` → `BTC-USD`
   - `oro` → `GC=F`

2. **Alias derivados del catálogo**  
   Se generan automáticamente a partir de la descripción de cada ticker del fichero base.

### Decisión de diseño

No queremos un diccionario infinito ni una lista imposible de mantener.  
La idea es usar:

- una base pequeña pero muy útil de alias manuales;
- y una capa automática derivada del catálogo.

### Resultado

Se genera un diccionario final `alias_to_ticker` que centraliza todas las equivalencias conocidas por el agente.

In [4]:
# ============================================================
# Construcción de alias
# ------------------------------------------------------------
# MANUAL_ALIAS_TO_TICKER:
#   Alias curados a mano para consultas frecuentes.
#
# ticker_row_to_aliases():
#   Genera alias automáticos a partir del catálogo.
#
# alias_to_ticker:
#   Diccionario final que concentra todos los alias conocidos
#   por el pipeline.
# ============================================================


MANUAL_ALIAS_TO_TICKER = {
    "sp500": "^GSPC",
    "sp 500": "^GSPC",
    "s p 500": "^GSPC",
    "s&p 500": "^GSPC",
    "s and p 500": "^GSPC",
    "dow jones": "^DJI",
    "nasdaq composite": "^IXIC",
    "russell 2000": "^RUT",
    "vix": "^VIX",
    "bitcoin": "BTC-USD",
    "btc": "BTC-USD",
    "ethereum": "ETH-USD",
    "eth": "ETH-USD",
    "solana": "SOL-USD",
    "oro": "GC=F",
    "gold": "GC=F",
    "plata": "SI=F",
    "silver": "SI=F",
    "petroleo": "CL=F",
    "petroleo crudo": "CL=F",
    "crude oil": "CL=F",
    "gas natural": "NG=F",
    "cobre": "HG=F",
    "eur usd": "EURUSD=X",
    "eur / usd": "EURUSD=X",
    "eur/usd": "EURUSD=X",
    "usd jpy": "JPY=X",
    "gbp usd": "GBPUSD=X",
    "aud usd": "AUDUSD=X",
    "apple": "AAPL",
    "microsoft": "MSFT",
    "alphabet": "GOOGL",
    "google": "GOOGL",
    "amazon": "AMZN",
    "meta": "META",
    "facebook": "META",
    "nvidia": "NVDA",
    "tesla": "TSLA",
    "intel": "INTC",
    "amd": "AMD",
    "broadcom": "AVGO",
    "qualcomm": "QCOM",
    "paypal": "PYPL",
    "jp morgan": "JPM",
    "jpmorgan": "JPM",
    "bank of america": "BAC",
    "coca cola": "KO",
    "pepsico": "PEP",
    "walmart": "WMT",
    "nike": "NKE",
}

STOP_ALIASES = {
    "stock", "index", "etf", "crypto", "commodity", "forex", "future", "futures",
    "large cap", "and", "the", "de", "el", "la", "los", "las", "y"
}
GENERIC_PREFIXES = {"large cap"}

def ticker_row_to_aliases(row: pd.Series) -> List[str]:
    aliases = [row["ticker"]]
    desc = row.get("description") or ""
    if desc:
        aliases.append(desc)
        core = re.sub(r"\(.*?\)", "", desc).strip()
        core = re.sub(r"\s+", " ", core)
        core_norm = normalize_text(core)
        if core_norm and core_norm not in STOP_ALIASES and not any(core_norm.startswith(pref) for pref in GENERIC_PREFIXES):
            aliases.append(core)

    clean = []
    for alias in aliases:
        alias_norm = normalize_text(alias)
        if alias_norm and alias_norm not in STOP_ALIASES and len(alias_norm) >= 2:
            clean.append(alias)
    return list(dict.fromkeys(clean))

catalog_by_ticker = {
    row["ticker"]: {
        "asset_type": row["asset_type"],
        "description": row["description"],
    }
    for _, row in catalog_df.iterrows()
}

auto_alias_to_ticker = {}
for _, row in catalog_df.iterrows():
    for alias in ticker_row_to_aliases(row):
        alias_norm = normalize_text(alias)
        if alias_norm not in auto_alias_to_ticker:
            auto_alias_to_ticker[alias_norm] = row["ticker"]

alias_to_ticker = dict(auto_alias_to_ticker)
for alias, ticker in MANUAL_ALIAS_TO_TICKER.items():
    alias_to_ticker[normalize_text(alias)] = ticker

KNOWN_TICKERS = set(catalog_df["ticker"].tolist())
DIRECT_TICKER_PATTERN = re.compile(r"\^[A-Z0-9.\-]+|[A-Z]{2,}[A-Z0-9.\-]*(?:=[A-Z])?")

len(alias_to_ticker), len(KNOWN_TICKERS)


(384, 188)

## 4. Intención mínima

Aunque este primer agente todavía no hace el análisis financiero final, sí conviene etiquetar **qué tipo de petición** ha hecho el usuario.

### Intenciones mínimas consideradas

- `price_growth`  
  Consultas del tipo “cuánto ha crecido…”.

- `compare_assets`  
  Consultas del tipo “compara Nvidia y AMD…”.

- `historical_download`  
  Consultas generales de descarga histórica.

### Por qué interesa clasificar intención

La intención no cambia necesariamente la descarga base, pero sí:
- mejora la trazabilidad;
- prepara el trabajo del segundo agente analista;
- deja abierta la puerta a políticas distintas más adelante.

In [5]:
# ============================================================
# Clasificación ligera de intención
# ------------------------------------------------------------
# Esta función no pretende entender toda la semántica de la
# consulta, sino asignar una etiqueta mínima que ayude a
# documentar la petición y preparar al segundo agente.
# ============================================================

def detect_intent(query: str) -> str:
    q = normalize_text(query)
    if any(k in q for k in ["compara", "comparar", "compare", " versus ", " vs "]):
        return "compare_assets"
    if any(k in q for k in ["ha crecido", "crecido", "evolucion", "evolución", "rendimiento", "revalorizacion"]):
        return "price_growth"
    return "historical_download"


## 5. Interpretación temporal

La parte temporal es una de las más importantes del pipeline, porque `yfinance` necesita una petición muy concreta.

### Política temporal aplicada

El pipeline distingue entre dos grandes casos:

- **fechas explícitas**  
  Por ejemplo: `desde 2024-01-01 hasta 2024-12-31`.

- **periodos relativos**  
  Por ejemplo: `5 años`, `3 meses`, `1 semana`.

### Qué salida produce

La interpretación temporal termina en uno de estos dos formatos:

- `start` / `end`
- `period`

### Regla práctica

- Si el usuario da fechas explícitas, se priorizan.
- Si no da fechas, pero sí una duración, se usa `period`.
- Si no especifica nada temporal, se activa una política por defecto con advertencia.

In [6]:
# ============================================================
# Interpretación temporal
# ------------------------------------------------------------
# Se definen:
# - alias de intervalos compatibles con yfinance;
# - equivalencias de unidades temporales;
# - funciones para detectar fechas explícitas, años y
#   periodos relativos.
#
# La salida de esta etapa es siempre:
#   (start, end, period, warnings)
# ============================================================

INTERVAL_ALIASES = {
    "1m": "1m",
    "2m": "2m",
    "5m": "5m",
    "15m": "15m",
    "30m": "30m",
    "60m": "60m",
    "90m": "90m",
    "1h": "1h",
    "1d": "1d",
    "1wk": "1wk",
    "1mo": "1mo",
    "3mo": "3mo",
    "diario": "1d",
    "daily": "1d",
    "semanal": "1wk",
    "weekly": "1wk",
    "mensual": "1mo",
    "monthly": "1mo",
}

SPANISH_UNITS_TO_YF = {
    "ano": "y",
    "anos": "y",
    "año": "y",
    "años": "y",
    "year": "y",
    "years": "y",
    "mes": "mo",
    "meses": "mo",
    "month": "mo",
    "months": "mo",
    "semana": "wk",
    "semanas": "wk",
    "week": "wk",
    "weeks": "wk",
    "dia": "d",
    "dias": "d",
    "día": "d",
    "días": "d",
    "day": "d",
    "days": "d",
}

INTRADAY_INTERVALS = {"1m", "2m", "5m", "15m", "30m", "60m", "90m", "1h"}

def find_interval(query: str) -> str:
    q = normalize_text(query)
    explicit = re.findall(r"\b(1m|2m|5m|15m|30m|60m|90m|1h|1d|1wk|1mo|3mo)\b", q)
    if explicit:
        return explicit[0]
    for alias, interval in INTERVAL_ALIASES.items():
        if normalize_text(alias) in q:
            return interval
    return "1d"

def find_explicit_dates(query: str) -> List[str]:
    return re.findall(r"\b\d{4}-\d{2}-\d{2}\b", query)

def infer_start_from_year(text: str) -> Optional[str]:
    m = re.search(r"\bdesde\s+(\d{4})\b", normalize_text(text))
    if m:
        return f"{m.group(1)}-01-01"
    return None

def find_relative_period(query: str) -> Optional[str]:
    q = normalize_text(query)
    m = re.search(
        r"(?:ultimos|ultimas|ultimo|ultima|last|en|de|durante|for)?\s*(\d+)\s*(anos|ano|años|año|years|year|meses|mes|months|month|semanas|semana|weeks|week|dias|dia|días|día|days|day)\b",
        q,
    )
    if not m:
        return None
    num = int(m.group(1))
    unit = m.group(2)
    return f"{num}{SPANISH_UNITS_TO_YF[unit]}"

def parse_temporal_expression(query: str) -> Tuple[Optional[str], Optional[str], Optional[str], List[str]]:
    warnings = []
    dates = find_explicit_dates(query)
    start = end = period = None

    if len(dates) >= 2:
        return dates[0], dates[1], None, warnings

    start_from_year = infer_start_from_year(query)
    if start_from_year:
        return start_from_year, None, None, warnings

    if len(dates) == 1:
        q = normalize_text(query)
        if "desde" in q:
            start = dates[0]
        elif "hasta" in q:
            end = dates[0]
        else:
            start = dates[0]
        return start, end, period, warnings

    period = find_relative_period(query)
    if period:
        return start, end, period, warnings

    warnings.append("No se detectó rango temporal explícito; se usa period='1y' por defecto.")
    return None, None, "1y", warnings


## 6. Resolución de activos

Esta es la parte que transforma el texto del usuario en símbolos válidos para Yahoo Finance.

### Estrategia de resolución

Se sigue una jerarquía simple y controlada:

1. Detectar **tickers directos** escritos por el usuario, como `AAPL`, `QQQ`, `^GSPC`.
2. Detectar **alias manuales**, como `nvidia`, `bitcoin`, `oro`.
3. Detectar **coincidencias derivadas del catálogo**.

### Resultado esperado

Cada activo identificado se convierte en un objeto `AssetResolution` con:
- texto original;
- ticker resuelto;
- tipo de activo;
- fuente de la resolución;
- confianza.

### Por qué es importante esta etapa

Aquí se produce el paso crítico de:

> nombre humano → ticker compatible con Yahoo Finance

In [7]:
# ============================================================
# Resolución de activos a ticker
# ------------------------------------------------------------
# find_direct_tickers_with_pos():
#   Busca símbolos ya escritos por el usuario.
#
# resolve_assets():
#   Combina tickers directos + alias manuales + alias del
#   catálogo y devuelve una lista de AssetResolution.
# ============================================================

def find_direct_tickers_with_pos(query: str) -> List[Tuple[str, int, str]]:
    candidates = []
    for m in DIRECT_TICKER_PATTERN.finditer(query):
        token = m.group(0)
        if token in KNOWN_TICKERS:
            candidates.append((token, m.start(), token))
    return candidates

def resolve_assets(query: str) -> List[AssetResolution]:
    q_norm = normalize_text(query)
    candidates = []

    # 1) Tickers directos
    for ticker, pos, matched_text in find_direct_tickers_with_pos(query):
        meta = catalog_by_ticker.get(ticker, {})
        candidates.append(
            {
                "raw_text": matched_text,
                "matched_text": matched_text,
                "ticker": ticker,
                "asset_type": meta.get("asset_type"),
                "description": meta.get("description"),
                "source": "direct_ticker",
                "confidence": 1.0,
                "pos": pos,
            }
        )

    # 2) Alias manuales + catálogo curado
    manual_alias_norms = {normalize_text(k) for k in MANUAL_ALIAS_TO_TICKER.keys()}

    for alias_norm, ticker in alias_to_ticker.items():
        for m in re.finditer(rf"\b{re.escape(alias_norm)}\b", q_norm):
            meta = catalog_by_ticker.get(ticker, {})
            candidates.append(
                {
                    "raw_text": alias_norm,
                    "matched_text": alias_norm,
                    "ticker": ticker,
                    "asset_type": meta.get("asset_type"),
                    "description": meta.get("description"),
                    "source": "alias_or_catalog",
                    "confidence": 0.95 if alias_norm in manual_alias_norms else 0.90,
                    "pos": m.start(),
                }
            )
            break

    # 3) Orden por aparición y deduplicación por ticker
    candidates = sorted(candidates, key=lambda x: (x["pos"], -len(x["matched_text"])))
    resolved = []
    seen_tickers = set()

    for c in candidates:
        if c["ticker"] in seen_tickers:
            continue
        resolved.append(
            AssetResolution(
                raw_text=c["raw_text"],
                matched_text=c["matched_text"],
                ticker=c["ticker"],
                asset_type=c["asset_type"],
                description=c["description"],
                source=c["source"],
                confidence=c["confidence"],
            )
        )
        seen_tickers.add(c["ticker"])

    return resolved


## 7. Validación

Una vez interpretada la consulta, no conviene descargar inmediatamente.  
Antes hay que **comprobar si la petición final es coherente**.

### Qué se valida

- Que haya al menos un activo resuelto.
- Que no se mezclen a la vez `period` y `start/end` sin criterio.
- Que el intervalo pedido sea compatible con el rango temporal estimado.

### Regla de negocio clave

Si el usuario pide un intervalo intradía (`1h`, `15m`, etc.) para un rango demasiado largo, el pipeline degrada la granularidad a `1d` y deja constancia de ello en `warnings`.

### Objetivo

Convertir una interpretación preliminar en una consulta:
- usable,
- consistente,
- y defendible técnicamente.

In [ ]:
# ============================================================
# Validación y normalización
# ------------------------------------------------------------
# Esta etapa aplica reglas de negocio sobre la FinanceQuery:
# - exige al menos un activo resuelto;
# - evita conflictos entre period y fechas explícitas;
# - controla incompatibilidades entre rango e intervalo.
# ============================================================

def estimate_days_from_period(period: Optional[str]) -> Optional[int]:
    if not period:
        return None
    m = re.fullmatch(r"(\d+)(y|mo|wk|d)", period)
    if not m:
        return None
    n = int(m.group(1))
    unit = m.group(2)
    factor = {"y": 365, "mo": 30, "wk": 7, "d": 1}[unit]
    return n * factor

def estimate_days_from_dates(start: Optional[str], end: Optional[str]) -> Optional[int]:
    try:
        if start and end:
            return (datetime.fromisoformat(end).date() - datetime.fromisoformat(start).date()).days
        if start and not end:
            return (TODAY - datetime.fromisoformat(start).date()).days
    except Exception:
        return None
    return None

def validate_and_normalize(fin: FinanceQuery) -> FinanceQuery:
    warnings = list(fin.warnings)

    if not fin.assets_resolved:
        fin.needs_clarification = True
        warnings.append("No se pudo resolver ningún activo a ticker compatible con Yahoo Finance.")

    if fin.period and (fin.start or fin.end):
        warnings.append("Se detectaron a la vez period y start/end; se priorizan fechas explícitas.")
        fin.period = None

    span_days = estimate_days_from_period(fin.period)
    if span_days is None:
        span_days = estimate_days_from_dates(fin.start, fin.end)

    if fin.interval in INTRADAY_INTERVALS and span_days is not None and span_days > 60:
        warnings.append(
            f"El intervalo '{fin.interval}' es intradía y el rango estimado ({span_days} días) supera 60 días. "
            "Se degrada el intervalo a '1d'."
        )
        fin.interval = "1d"

    fin.warnings = warnings
    return fin


## 8. Pipeline completo

Esta función junta todas las etapas anteriores y produce el objeto final `FinanceQuery`.

### Qué hace en orden

1. Detecta la intención.
2. Resuelve los activos.
3. Interpreta el intervalo.
4. Interpreta el rango temporal.
5. Construye el objeto `FinanceQuery`.
6. Lo valida y lo normaliza.

### Por qué esta función es importante

Es el verdadero **punto de entrada lógico** del primer agente.  
Todo el resto del notebook está al servicio de esta función.


In [ ]:
# ============================================================
# Constructor principal del pipeline
# ------------------------------------------------------------
# build_finance_query() es la puerta de entrada del agente:
# recibe texto libre y devuelve una FinanceQuery ya validada.
# ============================================================

def build_finance_query(user_query: str) -> FinanceQuery:
    intent = detect_intent(user_query)
    assets_resolved = resolve_assets(user_query)
    assets_raw = [a.raw_text for a in assets_resolved]

    interval = find_interval(user_query)
    start, end, period, temporal_warnings = parse_temporal_expression(user_query)

    fq = FinanceQuery(
        original_query=user_query,
        intent=intent,
        assets_raw=assets_raw,
        assets_resolved=assets_resolved,
        interval=interval,
        start=start,
        end=end,
        period=period,
        warnings=temporal_warnings,
    )
    return validate_and_normalize(fq)


## 9. Pruebas rápidas con varios ejemplos

Antes de descargar nada, es importante validar si la parte semántica funciona bien.

### Qué comprobamos aquí

Para varias consultas de ejemplo se observa:
- la intención detectada;
- los tickers resueltos;
- el formato temporal resultante;
- el intervalo final;
- si la consulta necesita aclaración;
- y las advertencias generadas.

### Objetivo de esta batería

Detectar errores conceptuales antes de introducir la complejidad de la descarga real.

In [ ]:
# ============================================================
# Batería de ejemplos
# ------------------------------------------------------------
# Estas pruebas permiten verificar si el pipeline se comporta
# de forma razonable antes de activar la descarga real.
# ============================================================

EXAMPLES = [
    "Cuánto ha crecido Nvidia en 5 años",
    "Descárgame el histórico del S&P 500 desde 2020",
    "Quiero el oro en 1 semana a 1h",
    "Compara Nvidia y AMD en 2 años",
    "Datos de Bitcoin desde 2024-01-01 hasta 2024-12-31",
    "QQQ y SPY desde 2024-01-01 hasta 2024-12-31",
    "AAPL en 3 meses",
    "EUR/USD en 10 días a 1h",
]

rows = []
for q in EXAMPLES:
    fq = build_finance_query(q)
    rows.append(
        {
            "query": q,
            "intent": fq.intent,
            "tickers": fq.resolved_tickers(),
            "start": fq.start,
            "end": fq.end,
            "period": fq.period,
            "interval": fq.interval,
            "needs_clarification": fq.needs_clarification,
            "warnings": " | ".join(fq.warnings) if fq.warnings else "",
        }
    )

pd.DataFrame(rows)


,query,intent,tickers,start,end,period,interval,needs_clarification,warnings
0,Cuánto ha crecido Nvidia en 5 años,price_growth,[NVDA],None,None,5y,1d,False,
1,Descárgame el histórico del S&P 500 desde 2020,historical_download,[^GSPC],2020-01-01,None,None,1d,False,
2,Quiero el oro en 1 semana a 1h,historical_download,[GC=F],None,None,1wk,1h,False,
3,Compara Nvidia y AMD en 2 años,compare_assets,"[NVDA, AMD]",None,None,2y,1d,False,
4,Datos de Bitcoin desde 2024-01-01 hasta 2024-12-31,historical_download,[BTC-USD],2024-01-01,2024-12-31,None,1d,False,
5,QQQ y SPY desde 2024-01-01 hasta 2024-12-31,historical_download,"[QQQ, SPY]",2024-01-01,2024-12-31,None,1d,False,
6,AAPL en 3 meses,historical_download,[AAPL],None,None,3mo,1d,False,
7,EUR/USD en 10 días a 1h,historical_download,[EURUSD=X],None,None,10d,1h,False,


In [17]:
# ============================================================
# Inspección detallada de un ejemplo concreto
# ------------------------------------------------------------
# Aquí se muestra el objeto FinanceQuery completo para poder
# revisar todos sus campos de forma transparente.
# ============================================================

example_query = "Compara Nvidia y AMD en 2 años"
fq = build_finance_query(example_query)
print(json.dumps(fq.to_dict(), indent=2, ensure_ascii=False))

{
  "original_query": "Compara Nvidia y AMD en 2 años",
  "intent": "compare_assets",
  "assets_raw": [
    "nvidia",
    "AMD"
  ],
  "assets_resolved": [
    {
      "raw_text": "nvidia",
      "matched_text": "nvidia",
      "ticker": "NVDA",
      "asset_type": "Stock",
      "description": "Nvidia",
      "source": "alias_or_catalog",
      "confidence": 0.95
    },
    {
      "raw_text": "AMD",
      "matched_text": "AMD",
      "ticker": "AMD",
      "asset_type": "Stock",
      "description": "AMD",
      "source": "direct_ticker",
      "confidence": 1.0
    }
  ],
  "interval": "1d",
  "start": null,
  "end": null,
  "period": "2y",
  "group_by": "ticker",
  "auto_adjust": false,
  "threads": true,
  "progress": false,
  "warnings": [],
  "needs_clarification": false,
  "resolved_tickers": [
    "NVDA",
    "AMD"
  ]
}


## 10. Construcción de la petición para `yf.download()`

Una vez tenemos un `FinanceQuery` correcto, construir la llamada técnica a `yfinance` debería ser un paso **mecánico y determinista**.

### Idea clave

El notebook separa claramente:

- la fase de **entender la consulta**;
- de la fase de **producir parámetros técnicos**.

### Qué sale de aquí

Un diccionario listo para usarse en:

```python
yf.download(**params)
```


In [ ]:

# ============================================================
# Construcción final de parámetros de descarga
# ------------------------------------------------------------
# A partir de una FinanceQuery válida, esta celda genera el
# diccionario exacto que se pasaría a yf.download(...).
# ============================================================

example_query = "Datos de Bitcoin desde 2024-01-01 hasta 2024-12-31"
fq = build_finance_query(example_query)
fq.to_download_params()

{'tickers': ['BTC-USD'],
 'interval': '1d',
 'group_by': 'ticker',
 'auto_adjust': False,
 'threads': True,
 'progress': False,
 'start': '2024-01-01',
 'end': '2024-12-31'}

## 11. Descarga real opcional

En esta fase del notebook, la descarga real queda **preparada pero desactivada por defecto**.

### Motivo

Primero interesa validar:
- la interpretación;
- la resolución de tickers;
- la normalización temporal;
- y las reglas de validación.

Solo después conviene activar la descarga real.

### Ventaja

Esto permite depurar el pipeline conceptual sin depender todavía de la red ni de Yahoo Finance.


In [18]:
# ============================================================
# Flag de control de descarga real
# ------------------------------------------------------------
# Mantener esta bandera en False permite depurar la lógica del
# agente sin depender todavía de conexión o de Yahoo Finance.
# ============================================================

DO_REAL_DOWNLOAD = True


In [22]:
# ============================================================
# Ejemplo de descarga real opcional
# ------------------------------------------------------------
# Si DO_REAL_DOWNLOAD = True, se lanza yf.download(...) con
# los parámetros construidos por el pipeline.
# ============================================================

if DO_REAL_DOWNLOAD:
    import yfinance as yf

    user_query = "Muestrame Apple desde 2015"
    fq = build_finance_query(user_query)
    raw = yf.download(**fq.to_download_params())

    print("Shape:", raw.shape)
    display(raw.head())
else:
    print("Descarga desactivada. Pon DO_REAL_DOWNLOAD = True para ejecutar yf.download(...).")


Shape: (2814, 6)


Ticker           AAPL                                                       
Price            Open       High        Low      Close  Adj Close     Volume
Date                                                                        
2015-01-02  27.847500  27.860001  26.837500  27.332500  24.214891  212818400
2015-01-05  27.072500  27.162500  26.352501  26.562500  23.532719  257142000
2015-01-06  26.635000  26.857500  26.157499  26.565001  23.534939  263188400
2015-01-07  26.799999  27.049999  26.674999  26.937500  23.864950  160423600
2015-01-08  27.307501  28.037500  27.174999  27.972500  24.781893  237458000

## 12. Postproceso

Una vez obtenidos los datos, el sistema debe guardar tanto el resultado bruto como el contexto de cómo se obtuvo.

### Qué se guarda

- el `CSV` con los datos devueltos por `yfinance`;
- un fichero `metadata.json` con la consulta normalizada;
- advertencias y decisiones del pipeline.

### Por qué es útil

Esto hace el sistema mucho más:
- reproducible;
- auditable;
- y fácil de documentar en el TFM.


In [ ]:
# ============================================================
# Guardado de resultados
# ------------------------------------------------------------
# slugify():
#   Genera nombres de fichero seguros a partir de la consulta.
#
# save_outputs():
#   Guarda el DataFrame bruto en CSV y la FinanceQuery en JSON
#   para conservar trazabilidad del experimento.
# ============================================================

def slugify(text: str) -> str:
    text = normalize_text(text).replace(" ", "_")
    text = re.sub(r"[^a-z0-9_\-]+", "", text)
    return text[:80].strip("_") or "query"

def save_outputs(raw_df: pd.DataFrame, finance_query: FinanceQuery, export_dir: Path = EXPORT_DIR) -> Dict[str, str]:
    export_dir.mkdir(exist_ok=True, parents=True)
    stem = slugify(finance_query.original_query)

    csv_path = export_dir / f"{stem}.csv"
    metadata_path = export_dir / f"{stem}.metadata.json"

    raw_df.to_csv(csv_path)
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(finance_query.to_dict(), f, ensure_ascii=False, indent=2)

    return {
        "csv_path": str(csv_path),
        "metadata_path": str(metadata_path),
    }


In [25]:
# ============================================================
# Ejemplo de postproceso completo
# ------------------------------------------------------------
# Esta celda muestra el flujo final:
# consulta -> descarga -> guardado de CSV y metadata.
# ============================================================

if DO_REAL_DOWNLOAD:
    import yfinance as yf

    user_query = "Comparame Nvidia, Apple y Google desde 2015"
    fq = build_finance_query(user_query)
    raw = yf.download(**fq.to_download_params())
    outputs = save_outputs(raw, fq)

    print(json.dumps(outputs, indent=2, ensure_ascii=False))
else:
    print("Guardado preparado. Activa DO_REAL_DOWNLOAD para generar CSV y metadata.")


{
  "csv_path": "exports\\comparame_nvidia_apple_y_google_desde_2015.csv",
  "metadata_path": "exports\\comparame_nvidia_apple_y_google_desde_2015.metadata.json"
}



## 13. Qué revisar ahora

Esta primera versión ya te permite trabajar sobre decisiones de diseño muy claras:

1. ¿Te convence la política de tiempo por defecto (`1y`) o prefieres exigir aclaración?
2. ¿Quieres degradar automáticamente un intradía inválido a `1d` o prefieres bloquear la consulta?
3. ¿Qué activos ambiguos quieres fijar por defecto (`oro` → `GC=F`, por ejemplo)?
4. ¿Qué alias extra quieres añadir a la capa curada?
5. ¿Cuándo quieres introducir búsqueda dinámica online?

## Siguiente paso recomendado

La siguiente iteración natural sería:
- mantener este pipeline estable;
- medir qué ejemplos resuelve bien;
- y añadir una capa opcional de resolución dinámica solo cuando un activo no se pueda resolver con catálogo + alias.
